# Análisis Exploratorio de Datos (EDA) - The Last Mile Gap

**Objetivo:** Documentar la exploración de los datos de Olist, validar la calidad del dataset y generar las visualizaciones base para responder a las 5 preguntas del Business Case.

**Autor:** Julio Sosa Angeles

In [1]:
import pandas as pd
import plotly.express as px
from sqlalchemy import create_engine
from dotenv import load_dotenv
import os

# Cargar entorno
load_dotenv()
DATABASE_URL = os.getenv("DATABASE_URL")
engine = create_engine(DATABASE_URL)

# Cargar la vista principal
query = "SELECT * FROM vw_order_delivery_metrics WHERE review_score IS NOT NULL"
df = pd.read_sql(query, engine)
print(f"Filas cargadas: {len(df)}")
df.head()

Filas cargadas: 96353


,order_id,customer_id,customer_state,order_purchase_timestamp,order_delivered_customer_date,order_estimated_delivery_date,review_score,dias_diferencia,is_late
0,73fc7af87114b39712e6da79b0a377eb,41dcb106f807e993532d446263290104,SP,2018-01-11 15:30:49,2018-01-17 18:42:41,2018-02-02,4,-15.0,0
1,8e6bfb81e283fa7e4f11123a3fb894f1,5986b333ca0d44534a156a52a8e33a83,SC,2018-02-10 10:59:03,2018-02-28 16:33:35,2018-03-09,5,-8.0,0
2,b18dcdf73be66366873cd26c5724d1dc,eecafc3ff695f031bfe354a9fff9d437,SP,2018-04-06 22:18:54,2018-04-12 17:17:53,2018-05-03,1,-20.0,0
3,e48aa0d2dcec3a2e87348811bcfdf22b,840be0730010e9580efd289bc9f37706,SP,2017-06-30 15:38:46,2017-07-15 12:57:24,2017-08-03,5,-18.0,0
4,c31a859e34e3adac22f376954e19b39d,98f485515e575607a8003b17e91990bd,RJ,2018-08-07 23:12:29,2018-08-13 18:08:28,2018-10-10,5,-57.0,0


## Pregunta 1: ¿Cuál es la relación entre el retraso y la reseña?
Vamos a agrupar por estado de retraso y calcular el promedio de reseña.

In [5]:
df_q1 = df.groupby('is_late')['review_score'].mean().reset_index()
df_q1['Estado'] = df_q1['is_late'].map({0: 'A tiempo', 1: 'Tardío'})

fig1 = px.bar(df_q1, x='Estado', y='review_score', color='Estado', text_auto='.2f',
              title="Impacto del retraso en la satisfacción",
              color_discrete_sequence=['#2ca02c', '#d62728'])
fig1.show()

## Pregunta 2: ¿Qué estados concentran la mayor proporción de retrasos?
Filtramos estados con más de 100 pedidos para evitar sesgos por bajo volumen.

In [4]:
df_q2 = df.groupby('customer_state')['is_late'].agg(['count', 'mean']).reset_index()
df_q2 = df_q2[df_q2['count'] > 100]
df_q2['porcentaje_tardios'] = df_q2['mean'] * 100
df_q2 = df_q2.sort_values(by='porcentaje_tardios', ascending=False).head(10)

fig2 = px.bar(df_q2, x='customer_state', y='porcentaje_tardios', 
              title="Top 10 Estados con mayor % de retrasos",
              labels={'porcentaje_tardios': '% Retrasos', 'customer_state': 'Estado'},
              color='porcentaje_tardios', color_continuous_scale='YlOrRd')
fig2.show()

## Conclusiones del EDA
1. Se confirma la hipótesis del Business Case: el retraso reduce la reseña de 4.29 a 2.57.
2. Existe una fuerte correlación geográfica: los estados del Nordeste (AL, MA, PI) son los más afectados.
3. La cantidad de ítems por pedido no sesga los resultados de retraso.
4. Los datos están limpios (0 duplicados) y los nulos de fecha de entrega (2.98%) fueron excluidos correctamente.